# 1. Import dependencies

In [30]:
!python -m pip install --quiet -r ../requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [31]:
import pandas as pd
import numpy as np

from churn_prediction.paths import PROCESSED_DATA_PATH

df = pd.read_csv(PROCESSED_DATA_PATH)

X = df.drop(
    columns=["customerID", "Churn"]
)
y = df["Churn"]

# 2. Split train, validation và test

## Split 2 tập train, validation set và test set

In [32]:
from sklearn.model_selection import train_test_split
X_dev, X_test, y_dev, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=69,
    stratify=y,
)

## Split 2 tập train set và validation set

In [33]:
X_train, X_val, y_train, y_val = train_test_split(
    X_dev,
    y_dev,
    test_size=0.2,
    random_state=67,
    stratify=y_dev,
)

## Kiểm tra

In [34]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(4507, 19)
(1127, 19)
(1409, 19)


# 3. Preprocessing

## Import thư viện

In [35]:
from sklearn.preprocessing import(
    OrdinalEncoder,
    OneHotEncoder,
    StandardScaler,
)

from sklearn.compose import ColumnTransformer

## Xác định loại của features

In [36]:
numerical_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]
yes_no_features = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling"
]
binary_features = [
    "SeniorCitizen",
    *yes_no_features
]
multiclass_features = [
    "gender",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaymentMethod"
]

## Kiểm tra

In [37]:
all_features = (
    numerical_features
    + binary_features
    + multiclass_features
)
assert len(all_features) == len(set(all_features))
assert set(all_features) == set(X.columns)

## Binary Transformer

In [38]:
binary_transformer = OrdinalEncoder(
    categories=[
        ["No", "Yes"]
        for _ in yes_no_features
    ]
)

## Preprocessor

In [39]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features,
        ),
        (
            "binary",
            binary_transformer,
            yes_no_features,
        ),
        (
            "senior",
            "passthrough",
            ["SeniorCitizen"],
        ),
        (
            "multi",
            OneHotEncoder(
                handle_unknown='ignore',
                sparse_output=False,    
            ),
            multiclass_features,
        )
    ],
    remainder="drop",
)

# 4. Dùng các model khác ngoài Logistic Regression 

## Import

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier,
)

## Define

In [41]:
models = {
    "LogisticRegression": LogisticRegression(
        max_iter=1000,
    ),
    "Decision Tree": DecisionTreeClassifier(
        random_state=36,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=36,
        n_jobs=-1,
    ),
    "HistGradientBoosting": HistGradientBoostingClassifier(
        random_state=36,
    ),
}

# 5. Pipeline

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline(
        steps=[
            (
                "preprocessor",
                clone(preprocessor),
            ),
            (
                "model",
                model,
            )
        ]
    )

# 6. Metrics

In [43]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
)

results = []

for name, pipeline in pipelines.items():
    pipeline.fit(
        X_train,
        y_train,
    )
    
    y_val_pred = pipeline.predict(
        X_val
    )
    y_val_prob_yes = pipeline.predict_proba(
        X_val
    )[:, 1]
    
    results.append({
        "model": name,
        
        "accuracy": accuracy_score(
            y_val,
            y_val_pred,
        ),
        "precision": precision_score(
            y_val,
            y_val_pred,
            pos_label="Yes",
        ),
        "recall": recall_score(
            y_val,
            y_val_pred,
            pos_label="Yes",
        ),
        "f1": f1_score(
            y_val,
            y_val_pred,
            pos_label="Yes",
        ),
        "roc_auc": roc_auc_score(
            y_val,
            y_val_prob_yes,
        ),
        "average_precision": average_precision_score(
            y_val,
            y_val_prob_yes,
            pos_label="Yes",
        )
    })

In [44]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "roc_auc",
    ascending=False,
)
results_df

,model,accuracy,precision,recall,f1,roc_auc,average_precision
0,LogisticRegression,0.811003,0.664122,0.581940,0.620321,0.849997,0.669032
1,Decision Tree,0.717835,0.470405,0.505017,0.487097,0.650584,0.369571
2,Random Forest,0.801242,0.666667,0.501672,0.572519,0.822050,0.624665
3,HistGradientBoosting,0.798580,0.647541,0.528428,0.581952,0.832378,0.631833


In [45]:
results_df.sort_values(
    "average_precision",
    ascending=False,
)

results_df

,model,accuracy,precision,recall,f1,roc_auc,average_precision
0,LogisticRegression,0.811003,0.664122,0.581940,0.620321,0.849997,0.669032
1,Decision Tree,0.717835,0.470405,0.505017,0.487097,0.650584,0.369571
2,Random Forest,0.801242,0.666667,0.501672,0.572519,0.822050,0.624665
3,HistGradientBoosting,0.798580,0.647541,0.528428,0.581952,0.832378,0.631833


## Tối ưu threshold

In [57]:
def find_best_f1_threshold(
    y_true,
    y_prob,
):
    rows = []
    
    for threshold in np.arange(
        0.1,
        0.91,
        0.01,
    ):
        y_pred = np.where(
            y_prob >= threshold,
            "Yes",
            "No",
        )
        rows.append({
            "threshold": threshold,
            
            "precision": precision_score(
                y_true,
                y_pred,
                pos_label="Yes",
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                pos_label="Yes",
                zero_division=0,
            ),
            "f1": f1_score(
                y_true,
                y_pred,
                pos_label="Yes",
                zero_division=0,
            ), 
        })
    threshold_df = pd.DataFrame(rows)
    
    return threshold_df.loc[
        threshold_df["f1"].idxmax()
    ]

In [58]:
threshold_results = []

for name, pipeline in pipelines.items():
    y_prob = pipeline.predict_proba(
        X_val
    )[:, 1]
    best = find_best_f1_threshold(
        y_val,
        y_prob
    )
    
    threshold_results.append({
        "model": name,
        "threshold": best["threshold"],
        "precision": best["precision"],
        "recall": best["recall"],
        "f1": best["f1"],
    })
threshold_results_df = pd.DataFrame(
    threshold_results
)

threshold_results_df

,model,threshold,precision,recall,f1
0,LogisticRegression,0.37,0.575835,0.749164,0.651163
1,Decision Tree,0.34,0.470588,0.508361,0.488746
2,Random Forest,0.23,0.467412,0.839465,0.600478
3,HistGradientBoosting,0.28,0.517621,0.785953,0.624170
